# 03 Models

The five networks of the study and the four single-change variants used in the ablations: their
design, their size, a forward pass on a real validation window, and the check that every port matches
its original network. CPU only; nothing is trained.

In [1]:
import os
import subprocess
import sys
import time

import numpy as np
import torch

sys.path.insert(0, os.path.abspath('../src'))
import local

S = local.load_study()
C = S.C
torch.set_num_threads(max(1, os.cpu_count() - 1))

device: cpu  backbone: resgru


## 1. The networks

| Network | Design | Inductive bias |
|:--|:--|:--|
| ResGRU-UNet | all 10 hours stacked as 200 channels; conv U-Net to 18 x 16; ConvGRU unrolled over the 16 lead hours; per-lead decoder with full-resolution skips | multi-scale convolution; a recurrent state that carries the forecast from hour to hour |
| AFNO V4 | each hour encoded by shared conv blocks; one AFNO block at 35 x 31 and six at 18 x 16; a 1-D temporal translator from 10 to 16 hours; two AFNO blocks per lead hour; U-Net decoder whose skips keep the last hour and the mean | global spectral mixing in every block |
| FMG-WACLU | ConvLSTM U-Net with wind speed, direction and divergence channels, gated by weather rolled forward by a second ConvLSTM | recurrence over the input hours plus wind physics |
| ConvLSTM | ConvLSTM over the 10 input hours at full resolution, an FNO-style spectral block, and a shift of the PM2.5 map along the wind | recurrence plus a semi-Lagrangian transport step |
| patch AFNO | 4 x 4 patches, AFNO blocks, one linear layer back to the grid | global spectral mixing on patches |

Every network outputs a change added to the last observed hour in $\log(1+y)$, with a zero-initialized
last layer, so an untrained network is exactly persistence.

In [2]:
counts = {}
nets = ['resgru', 'afnov4', 'fmgwaclu', 'convlstm', 'patchafno', 'afnov4_allskip', 'afnov4_pad', 'afnov4_nosparse', 'resgru_norec']
logged = {'resgru': 'resgru', 'fmgwaclu': 'fmgwaclu', 'convlstm': 'convlstm', 'patchafno': 'patchafno'}
for name in nets:
    torch.manual_seed(0)
    model = S.BACKBONES[name]()
    counts[name] = sum(p.numel() for p in model.parameters())
    note = ''
    if name in logged:
        note = f"(Kaggle run logged {local.load_json(logged[name], 'summary.json')['params']:,})"
    print(f'{name:16s} {counts[name]:>11,} parameters {note}')

resgru             7,122,113 parameters (Kaggle run logged 7,122,113)


afnov4             6,980,033 parameters 
fmgwaclu           3,454,645 parameters (Kaggle run logged 3,454,645)
convlstm          19,001,840 parameters (Kaggle run logged 19,001,840)


patchafno          5,708,800 parameters (Kaggle run logged 5,708,800)


afnov4_allskip     7,668,161 parameters 
afnov4_pad         6,980,033 parameters 


afnov4_nosparse    6,980,033 parameters 
resgru_norec       7,126,977 parameters 


The variants keep their family's size: H1 (all-hour skips) adds the wider fusion layers, H2 (padding)
and H3 (no soft threshold) change no weights, and H4 replaces the ConvGRU by per-lead convolution
blocks of nearly the same size.

## 2. A forward pass on a real validation window

At initialization every network must return persistence exactly, because its last layer is zero.

In [3]:
stats = S.build_stats()
months, static = S.load_months(stats)
val = S.PM25Windows(months, static, stats, 'val')
x, t_log, t_raw, last_log, mask = val[0]
del months
print('input', tuple(x.shape), '-> output (lead hours, rows, columns)')
for name in nets:
    torch.manual_seed(0)
    model = S.BACKBONES[name]().eval()
    with torch.no_grad():
        t0 = time.time()
        out = model(x[None], last_log[None])[0]
        dt = time.time() - t0
    diff = (out - last_log[None]).abs().max().item()
    print(f'{name:16s} {tuple(out.shape)}  max |output - persistence| = {diff:.1e}  CPU {dt:.2f} s')

  APRIL_16: T=715  ep_ratio=0.0208


  JULY_16: T=739  ep_ratio=0.0197


  OCT_16: T=739  ep_ratio=0.0170


  DEC_16: T=739  ep_ratio=0.0217
  val hour estimation: <=1h 0.977
  val: 214 windows  ep_density mean=0.0182
input (20, 10, 140, 124) -> output (lead hours, rows, columns)


resgru           (16, 140, 124)  max |output - persistence| = 0.0e+00  CPU 0.31 s


afnov4           (16, 140, 124)  max |output - persistence| = 0.0e+00  CPU 1.35 s


... (2 outputs left out)

afnov4_allskip   (16, 140, 124)  max |output - persistence| = 0.0e+00  CPU 1.32 s


afnov4_pad       (16, 140, 124)  max |output - persistence| = 0.0e+00  CPU 1.50 s


afnov4_nosparse  (16, 140, 124)  max |output - persistence| = 0.0e+00  CPU 1.30 s


resgru_norec     (16, 140, 124)  max |output - persistence| = 0.0e+00  CPU 0.33 s


## 3. Port check

Each network was ported from its original Kaggle notebook (`reference/originals/`). The check rebuilds
every original from its notebook, confirms its logged parameter count, and confirms that the port
gives exactly the same output for the same weights. Only the input width and, for ConvLSTM and patch
AFNO, the residual output changed in the study versions; patch AFNO also had its spectral bug fixed.

In [4]:
r = subprocess.run([sys.executable, local.path('tests', 'local_check.py'), local.path('reference', 'originals')],
                   capture_output=True, text=True, cwd=local.ROOT)
lines = [l for l in r.stdout.splitlines() if 'PASS' in l or 'FAIL' in l or 'ALL CHECKS' in l]
print('\n'.join(lines))
print('exit code', r.returncode)

  PASS resgru original params 7,096,513 == logged 7,096,513
  PASS resgru port params 7,096,513 == original
  PASS resgru port output == original (max diff 0.00e+00)
  PASS fmgwaclu original params 3,451,057 == logged 3,451,057
  PASS fmgwaclu port params 3,451,057 == original
  PASS fmgwaclu port output == original (max diff 0.00e+00)
  PASS convlstm original params 19,063,792 == logged 19,063,792
  PASS convlstm port params 18,997,232 == original minus season FiLM (66,560)
  PASS convlstm port has every original weight except season FiLM (missing [], extra ['film_bi', 'film_sc', 's_emb'])
  PASS convlstm port output == original with FiLM set to identity (max diff 0.00e+00)
  PASS patchafno original params 5,422,080 == logged 5,422,080
  PASS patchafno port params 5,422,080 == original
  PASS patchafno port output == original with only xf.float() -> xf (max diff 0.00e+00)
  PASS patchafno: the xf.float() line is the only difference
  PASS afnov4 params 6,980,033 == afno_v4.py
  PASS a

## 4. How far each network looks

Over 16 hours PM2.5 travels about 10 cells (notebook 01). A $3\times3$ convolution reaches one more cell
per layer: ResGRU-UNet's encoder reaches about 14 cells and each ConvGRU step at $18\times16$ adds about 8
more. An AFNO block mixes the whole map through learned spectral weights in every block, and its FFT
treats opposite edges as neighbours.

Strictly, both networks depend on every input cell: GroupNorm computes statistics over the whole map,
and ResGRU-UNet's bottleneck and 16 ConvGRU steps also span it. The difference is how much weight
nearby cells get. Below, the gradient of one output cell (lead hour 1, map centre) with respect to the
input shows how its sensitivity is spread, for slightly perturbed **untrained** weights; trained
networks may differ.

In [5]:
def gradient_map(model, x, last_log, i=70, j=62):
    x = x[None].clone().requires_grad_(True)
    out = model(x, last_log[None])[0, 0]
    out[i, j].backward()
    return x.grad[0].abs().sum(dim=(0, 1)).numpy()


yy, xx = np.mgrid[0:C.H, 0:C.W]
dist = np.hypot(yy - 70, xx - 62)
for name in ('resgru', 'afnov4'):
    torch.manual_seed(0)
    model = S.BACKBONES[name]()
    with torch.no_grad():
        for p in model.parameters():
            p.add_(0.01 * torch.randn_like(p))
    g = gradient_map(model.eval(), x, last_log)
    print(f'{name:8s} depends on {int((g > 0).sum()):,} of {C.H * C.W:,} input cells; share of the gradient within '
          + ', '.join(f'{r} cells: {100 * g[dist <= r].sum() / g.sum():.1f}%' for r in (3, 12, 30)))

resgru   depends on 17,360 of 17,360 input cells; share of the gradient within 3 cells: 27.9%, 12 cells: 72.2%, 30 cells: 87.0%


afnov4   depends on 17,360 of 17,360 input cells; share of the gradient within 3 cells: 18.3%, 12 cells: 72.1%, 30 cells: 84.9%


Before training, both networks put about 72% of this sensitivity within 12 cells, and ResGRU-UNet
concentrates more of it on the nearest cells (28% against 18% within 3 cells). So, at least at
initialization, the two differ in degree rather than in kind: both can reach the whole map, and both
weight nearby cells most. The 10-cell transport distance is covered by either design.